# **Lesson_5.1**

## In this lecture

* Fork repository

* K-means clustering: cluster prediction for an individual input
* How to save and reuse your trained model
* In-class exercise: clustering helper function
* Linear regression code-along walkthrough project **Medical Charges**
* In-class exercise: medical data clustering

---

## Cluster prediction for an individual input

#### Prepare input
The new person (customer) is:
* Age: 30
* Annual income: 60k
* Spending score: 50

#### Create input for this customer:

In [ ]:
import pandas as pd
import joblib

In [ ]:
new_customer_df = pd.DataFrame([[30, 60, 50]], columns=['Age', 'Annual_Income', 'Spending_Score'])  # N.b.: _2D_array_
new_customer_df

#### Predict the cluster

In [ ]:
kmeans = joblib.load("../models/kmeans_v1.pkl")
kmeans

In [ ]:
cluster_label = kmeans.predict(new_customer_df)
print(f"The customer belongs to cluster: {cluster_label[0]}")


<fieldset>
<legend>DANGER ZONE</legend>
If you apply any preprocessing to your original dataset (like PCA, scaling ...), you should apply the same to the <b>new_customer_df</b> otherwise prediction will be wrong!
</fieldset>

In [ ]:
distances = kmeans.transform(new_customer_df)
print("Distances to cluster centers:", distances)

### In-class exercise

Write a helper function which would ask a user to enter *age*, *annual income* and *spending score* and **return** which *cluster* the new customer belongs to.

In [ ]:
# Write your code here ...
import pandas as pd
import joblib
age = input('Enter age?')
annual_income= input('Enter annual income?')
spending_score= input('Enter spending score?')
new_customer_df = pd.DataFrame([[age, annual_income, spending_score]], columns=['Age', 'Annual_Income', 'Spending_Score'])  # N.b.: _2D_array_
new_customer_df
kmeans = joblib.load("../models/kmeans_v1.pkl")
kmeans
cluster_label = kmeans.predict(new_customer_df)
print(f"The customer belongs to cluster: {cluster_label[0]}")


---

## Linear regression model for Medical Charges prediction

### Business objectives

* Purpose of the project: Predicting medical expences using Linear Regression
* Business question: what would be medical charges for new customers?

### Import and settings

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px # Interactive charts and save some coding; .express - high-level api
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Change settings to improve default style (optional)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

### Load data

In [ ]:
# Data path
data_path = '../datasets/medical-charges.csv'

# Load data
medical_df = pd.read_csv(data_path)

### EDA

In [ ]:
medical_df.info()

In [ ]:
medical_df.describe()

In [ ]:
medical_df.head()

#### Visualisation

* Age

In [ ]:
fig = px.histogram(
    medical_df,
    x='age',
    marginal='box',
    nbins=47, # bin for each year. Calculated from min and max age in the dataset.
    title="Age distribution"
)
fig.update_layout(bargap=0.1)
fig.show()

* BMI

In [ ]:
fig = px.histogram(
    medical_df,
    x='bmi',
    marginal='box',
    color_discrete_sequence=['red'],
    nbins=47, # bin for each year. Calculated from min and max age in the dataset.
    title="BMI distribution"
)
fig.update_layout(bargap=0.1)
fig.show()

* Charges

In [ ]:
fig = px.histogram(
    medical_df,
    x='charges',
    # color='smoker',
    marginal='box',
    # color_discrete_sequence=['green', 'grey'],
    nbins=47, # bin for each year. Calculated from min and max age in the dataset.
    title="Annual medical charges"
)
fig.update_layout(bargap=0.1)
fig.show()

* Smoker

In [ ]:
medical_df.smoker.value_counts()

In [ ]:
fig = px.histogram(
    medical_df,
    x='smoker',
    color='sex',
    title="Smoker"
)

fig.show()

* Age and charges

In [ ]:
fig = px.scatter(
    medical_df,
    x='age',
    y='charges',
    color='smoker',
    opacity=0.8,
    hover_data=['sex'],
    title="Age vs. Charges"    
)
fig.update_traces(marker_size=5)
fig.show()

* BMI and charges

In [ ]:
fig = px.scatter(
    medical_df,
    x='bmi',
    y='charges',
    color='smoker',
    opacity=0.8,
    hover_data=['sex'],
    title="BMI vs. Charges"    
)
fig.update_traces(marker_size=5)
fig.show()

* Number of children

In [ ]:
fig = px.violin(  # Violing used as an example
    medical_df,
    x='children',
    y='charges'
)
fig.show()

### In-class exercise
Before we proceed with building **linear regression model** to predict medical charges, apply k-means clustering using all applicable numerical features in the dataset (reuse the workflow we studied in the previous lesson).

In [ ]:
from sklearn.cluster import KMeans
data_path = '../datasets/medical-charges.csv'

# Load data
df = pd.read_csv(data_path)

Y = df[['age', 'bmi']]

wscc_ab = []
for i in range(1, 11):
    kmean_ab = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmean_ab.fit(Y)
    wscc_ab.append(kmean_ab.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wscc_ab)
plt.xlabel("Number of clusters")
plt.ylabel("WCSS")
plt.title("Cluster number optimisation by elbow")
plt.show()

kmean_ab = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)

y_ab_kmeans = kmeans.fit_predict(Y)
df['Cluster'] = y_ab_kmeans
df.head()

plt.figure(figsize=(10, 6))
plt.scatter(Y.iloc[:,0], Y.iloc[:,1], c=y_ab_kmeans, s=150, cmap='viridis')  # review slicing through iterables; 0, 1 stands for columns
centers = kmeans.cluster_centers_  # Retrieves coordinates of cluster centers
plt.scatter(centers[:,0], centers[:,1], c='red', s=200, alpha=.75, marker='X')
plt.xlabel("Age")
plt.ylabel("BMI")
plt.title("Patient clusters")
plt.show()


In [ ]:
A = df[['age', 'charges']]

wscc_ac = []
for i in range(1, 11):
    kmean_ac = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmean_ac.fit(A)
    wscc_ac.append(kmean_ac.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wscc_ac)
plt.xlabel("Number of clusters")
plt.ylabel("WCSS")
plt.title("Cluster number optimisation by elbow")
plt.show()

kmean_ac = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)

y_ac_kmeans = kmean_ac.fit_predict(A)
df['Cluster'] = y_ac_kmeans
df.head()

plt.figure(figsize=(10, 6))
plt.scatter(A.iloc[:,0], A.iloc[:,1], c=y_ac_kmeans, s=150, cmap='viridis')  # review slicing through iterables; 0, 1 stands for columns
centers = kmean_ac.cluster_centers_  # Retrieves coordinates of cluster centers
plt.scatter(centers[:,0], centers[:,1], c='red', s=200, alpha=.75, marker='X')
plt.xlabel("Age")
plt.ylabel("BMI")
plt.title("Patient clusters")
plt.show()


---

##### Reminder: do not forget to **Clear All Outputs**
### Now you can commit and push your code to **GitHub**